[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/05_ONNX_Operators_and_OpSets/02_OpSet_Versions/OpSet_Versions_Deep_Dive.ipynb)

# 5.2 OpSet Versions and Domains — Deep Dive

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [What is an OpSet?](#1-what-is-an-opset) | Formal definition, versioned domain-scoped collection |
| 2 | [Default Domain vs Custom Domains](#2-default-domain-vs-custom-domains) | Domain hierarchy and namespacing |
| 3 | [Version Resolution Algorithm](#3-version-resolution-algorithm) | How runtimes resolve operator schemas |
| 4 | [since_version Tracking](#4-since-version-tracking) | When operators were introduced or modified |
| 5 | [Semantic Changes Across Versions](#5-semantic-changes-across-versions) | Breaking changes (e.g., Softmax at opset 13) |
| 6 | [Version Converter](#6-version-converter) | Upgrade/downgrade models between opsets |
| 7 | [Compatibility Triangle](#7-compatibility-triangle) | Model requirements vs runtime capabilities |
| 8 | [Multi-Domain Models](#8-multi-domain-models) | Mixing default and custom domains |
| 9 | [IR Version vs OpSet Version](#9-ir-version-vs-opset-version) | Two independent version axes |
| 10 | [Practical Checklist for OpSet Selection](#10-practical-checklist) | Decision guide |
| 11 | [Key Takeaways](#11-key-takeaways) | Summary |

In [ ]:
# !pip install onnx numpy matplotlib --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, checker, numpy_helper, defs
from onnx import shape_inference, version_converter
import matplotlib.pyplot as plt

print(f"ONNX version:  {onnx.__version__}")
print(f"Default opset: {defs.onnx_opset_version()}")
print(f"IR version:    {onnx.IR_VERSION}")

<a id='1-what-is-an-opset'></a>
## 1. What is an OpSet?

An **operator set (OpSet)** is a versioned, domain-scoped collection of operator definitions
that specifies the exact semantics available to a model. Formally:

$$\text{OpSet} = (\texttt{domain}, \; \texttt{version}) \;\mapsto\; \{\text{schema}_1, \text{schema}_2, \ldots, \text{schema}_n\}$$

Each schema defines one operator at one version: its inputs, outputs, attributes, type
constraints, and semantic behavior.

### How Models Declare OpSets

Every ONNX `ModelProto` carries explicit **opset imports** that declare which operator
definitions apply:

```
┌──────────────────────────────────────────────────────────────┐
│                        ModelProto                            │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  opset_import[0]:                                            │
│    domain  = ""          ← default ONNX domain               │
│    version = 17          ← selects opset 17 schemas          │
│                                                              │
│  opset_import[1]:                                            │
│    domain  = "ai.onnx.ml"                                    │
│    version = 3           ← ML domain at version 3            │
│                                                              │
│  opset_import[2]:                                            │
│    domain  = "com.myorg.custom"                               │
│    version = 1           ← custom domain at version 1        │
│                                                              │
│  ir_version = 8          ← independent from opset            │
│                                                              │
├──────────────────────────────────────────────────────────────┤
│  GraphProto:                                                 │
│    Node(op="Conv",    domain="")        → uses opset 17      │
│    Node(op="TreeEnsembleClassifier",                         │
│         domain="ai.onnx.ml")           → uses opset 3        │
│    Node(op="MyFusedOp",                                      │
│         domain="com.myorg.custom")     → uses opset 1        │
└──────────────────────────────────────────────────────────────┘
```

### Key Invariant

OpSet versions are **monotonically increasing** integers. Once an operator schema is
published at version $v$, its contract at that version is **frozen forever**. New behavior
requires a new version.

In [ ]:
# Inspect opset imports in a model
x = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 10])
y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 10])
node = helper.make_node("Relu", ["X"], ["Y"])
graph = helper.make_graph([node], "demo", [x], [y])

# Build models at different opset versions
print(f"{'Opset':>6} │ {'IR Version':>11} │ {'Valid':>5} │ Notes")
print("─" * 55)
for opset in [7, 9, 11, 13, 15, 17, 18, 19, 20, 21]:
    try:
        m = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])
        checker.check_model(m)
        print(f"{opset:>6} │ {m.ir_version:>11} │ {'Yes':>5} │ Relu since_version=1")
    except Exception as e:
        print(f"{opset:>6} │ {'?':>11} │ {'No':>5} │ {str(e)[:40]}")

<a id='2-default-domain-vs-custom-domains'></a>
## 2. Default Domain vs Custom Domains

ONNX uses a **domain-based namespace** system to organize operators:

| Domain | Convention | Description | Example Ops |
|--------|-----------|-------------|-------------|
| `""` (empty) | Default | Standard ONNX inference operators | Conv, MatMul, Relu |
| `"ai.onnx"` | Canonical | Same as empty (interchangeable) | Conv, MatMul, Relu |
| `"ai.onnx.ml"` | ML | Classical ML operators | TreeEnsemble, SVMClassifier |
| `"ai.onnx.training"` | Training | Training-specific ops | Gradient, Adam |
| `"ai.onnx.preview.training"` | Preview | Experimental training ops | (deprecated) |
| `"com.microsoft"` | Vendor | Microsoft ORT extensions | FusedGemm, BiasGelu |
| `"com.myorg.*"` | Custom | User-defined domains | Any custom op |

### Domain Precedence Rules

When a `NodeProto` specifies `domain=""` (or omits it), the runtime looks up the schema
in the default ONNX operator set at the version declared in `opset_import`. Each domain
is versioned **independently**:

```
  Model declares:
    opset_import["" ] = 17     ← Conv, Relu, etc. at opset 17
    opset_import["ai.onnx.ml"] = 3   ← TreeEnsemble at opset 3

  Node(op_type="Conv", domain="")      → resolved from opset 17
  Node(op_type="TreeEnsembleClassifier",
       domain="ai.onnx.ml")            → resolved from ML opset 3
```

In [ ]:
# Enumerate available domains and their operators
all_schemas = defs.get_all_schemas_with_history()

domain_ops = {}
for schema in all_schemas:
    d = schema.domain if schema.domain else '"" (default)'
    domain_ops.setdefault(d, set()).add(schema.name)

print(f"{'Domain':<30} │ {'Unique Ops':>10} │ Examples")
print("─" * 80)
for domain, ops in sorted(domain_ops.items()):
    examples = ', '.join(sorted(ops)[:4])
    suffix = f', ... (+{len(ops)-4})' if len(ops) > 4 else ''
    print(f"{domain:<30} │ {len(ops):>10} │ {examples}{suffix}")

# Show the latest opset version for default domain
latest_opset = defs.onnx_opset_version()
print(f"\nLatest default opset version: {latest_opset}")

# Count schemas per version
version_counts = {}
for schema in all_schemas:
    if schema.domain == '' or schema.domain == 'ai.onnx':
        version_counts[schema.since_version] = version_counts.get(schema.since_version, 0) + 1

print(f"\nSchemas introduced/updated per opset version:")
for v in sorted(version_counts.keys()):
    bar = '█' * (version_counts[v] // 2)
    print(f"  Opset {v:>2}: {version_counts[v]:>3} schemas {bar}")

<a id='3-version-resolution-algorithm'></a>
## 3. Version Resolution Algorithm

When a runtime encounters a node with `(op_type=t, domain=d)`, it must resolve which
schema version to use. The algorithm finds the **highest schema version that does not
exceed the imported opset version**.

### Formal Definition

Let $v_d$ be the opset version imported for domain $d$. The effective schema version is:

$$v_{\text{eff}}(t, d) = \max\{v_s \mid v_s \leq v_d \;\wedge\; \text{schema\_exists}(t, v_s, d)\}$$

where $v_s$ is any `since_version` at which operator $t$ was defined or updated in domain $d$.

### Resolution Walkthrough

```
  Model imports opset version 15 for default domain
  Node uses op_type = "Softmax"

  Softmax schema history:
    since_version = 1   (original definition)
    since_version = 11  (updated: axis handling change)
    since_version = 13  (updated: axis=last-dim by default)

  Available versions ≤ 15:  {1, 11, 13}
  v_eff = max{1, 11, 13} = 13

  → Runtime uses Softmax schema from opset 13
```

### Edge Cases

- If no schema exists at or below $v_d$, the model is **invalid** (operator not available).
- If the op was deprecated, the last valid schema still applies at its version.
- For **unknown domains** (custom ops), the runtime may skip schema validation entirely.

### Formal Resolution Table

For operator $t$ with versions $\{v_1, v_2, \ldots, v_k\}$ where $v_1 < v_2 < \ldots < v_k$:

$$v_{\text{eff}}(t, d) = \begin{cases}
\text{undefined} & \text{if } v_d < v_1 \\
v_i & \text{if } v_i \leq v_d < v_{i+1} \\
v_k & \text{if } v_d \geq v_k
\end{cases}$$

In [ ]:
# Implement and demonstrate the version resolution algorithm
def resolve_schema_version(op_type, imported_opset, domain=""):
    """Resolve the effective schema version for an operator."""
    all_schemas = defs.get_all_schemas_with_history()
    versions = sorted(set(
        s.since_version for s in all_schemas
        if s.name == op_type and (s.domain == domain or (s.domain == 'ai.onnx' and domain == ''))
    ))
    if not versions:
        return None, []
    candidates = [v for v in versions if v <= imported_opset]
    if not candidates:
        return None, versions
    return max(candidates), versions

# Demonstrate resolution for several operators across opset versions
test_ops = ["Relu", "Softmax", "Reshape", "Resize", "LayerNormalization"]
test_opsets = [7, 9, 11, 13, 15, 17, 19]

print(f"{'Operator':<22} │ {'Versions':>25} │ Resolution at imported opset")
print("─" * 90)
for op in test_ops:
    _, all_versions = resolve_schema_version(op, 99)
    resolutions = []
    for opset in test_opsets:
        v_eff, _ = resolve_schema_version(op, opset)
        resolutions.append(f"{opset}→{v_eff}" if v_eff else f"{opset}→N/A")
    ver_str = str(all_versions)
    res_str = '  '.join(resolutions)
    print(f"{op:<22} │ {ver_str:>25} │ {res_str}")

<a id='4-since-version-tracking'></a>
## 4. since_version Tracking

Every operator schema records a `since_version` — the opset version at which that
particular schema was introduced or last modified. This is the **minimum opset version**
needed to use that specific behavior.

### Why Operators Get New Versions

Operators receive new `since_version` entries when:

1. **New attributes** are added (e.g., ReduceSum gains `noop_with_empty_axes` at opset 18)
2. **Semantic changes** occur (e.g., Softmax axis behavior at opset 13)
3. **Type support** is expanded (e.g., adding bfloat16 support)
4. **Input/output changes** (e.g., ReduceSum axes move from attribute to input at opset 13)

### Version History Patterns

```
  ReduceSum version history:
  ┌─────────┬───────────────────────────────────────────────┐
  │ Opset 1 │ axes as attribute, keepdims as attribute      │
  ├─────────┼───────────────────────────────────────────────┤
  │ Opset 11│ Minor type constraint updates                 │
  ├─────────┼───────────────────────────────────────────────┤
  │ Opset 13│ axes moved from ATTRIBUTE to INPUT            │
  │         │ → enables dynamic axis selection               │
  ├─────────┼───────────────────────────────────────────────┤
  │ Opset 18│ noop_with_empty_axes attribute added          │
  └─────────┴───────────────────────────────────────────────┘
```

In [ ]:
# Trace version history for key operators
def get_version_history(op_type, domain=""):
    """Get all since_versions for an operator."""
    all_schemas = defs.get_all_schemas_with_history()
    versions = []
    for s in all_schemas:
        if s.name == op_type and (s.domain == domain or (s.domain == 'ai.onnx' and domain == '')):
            versions.append(s.since_version)
    return sorted(set(versions))

# Track important operators
tracked_ops = [
    "Relu", "Conv", "MatMul", "Gemm", "Softmax", "BatchNormalization",
    "Reshape", "Squeeze", "Unsqueeze", "ReduceSum", "ReduceMean",
    "Resize", "Pad", "Slice", "Split", "LayerNormalization",
    "GroupNormalization", "Gelu", "Flatten", "Transpose",
]

print(f"{'Operator':<25} │ {'since_versions':>30} │ {'Updates':>7}")
print("─" * 70)
for op in sorted(tracked_ops):
    versions = get_version_history(op)
    if versions:
        ver_str = ', '.join(str(v) for v in versions)
        print(f"{op:<25} │ {ver_str:>30} │ {len(versions):>7}")
    else:
        print(f"{op:<25} │ {'(not found)':>30} │ {0:>7}")

# Find operators with the most version updates
all_schemas = defs.get_all_schemas_with_history()
op_update_counts = {}
for s in all_schemas:
    if s.domain == '' or s.domain == 'ai.onnx':
        op_update_counts[s.name] = op_update_counts.get(s.name, 0) + 1

print(f"\nTop 10 most-updated operators:")
for op, count in sorted(op_update_counts.items(), key=lambda x: -x[1])[:10]:
    versions = get_version_history(op)
    print(f"  {op:<25} {count} versions: {versions}")

<a id='5-semantic-changes-across-versions'></a>
## 5. Semantic Changes Across Versions

Some opset version bumps introduce **semantic changes** that alter operator behavior.
These are the most important changes to understand:

### Softmax Axis Change (Opset 13)

**Before opset 13** (opset 1–12): the `axis` attribute split the input into a 2D matrix
$[\text{left-of-axis}, \text{right-of-axis}]$, then applied softmax over the right portion.

**From opset 13**: the `axis` attribute specifies a **single axis** for softmax computation,
matching PyTorch/TensorFlow behavior. Default changed to `axis=-1` (last dimension).

```
  Input shape: [2, 3, 4]

  Opset 12, axis=1:  flatten to [2, 12] → softmax over dim 1 → reshape back
  Opset 13, axis=1:  softmax over dim 1 only (size 3) → output [2, 3, 4]

  ⚠ DIFFERENT RESULTS for the same axis value!
```

### ReduceSum: Attribute → Input (Opset 13)

**Before opset 13**: `axes` was an **attribute** (fixed at graph construction time).

**From opset 13**: `axes` became a **second input** (can be computed dynamically at runtime).

### Squeeze/Unsqueeze: Attribute → Input (Opset 13)

Similarly, `axes` moved from attribute to input at opset 13, enabling dynamic axis selection.

### Resize: Coordinate Transform (Opset 11)

Opset 11 introduced the `coordinate_transformation_mode` attribute, replacing the simpler
opset 10 semantics. This affects how pixel coordinates are mapped during upsampling/downsampling.

### Summary of Major Breaking Changes

| Opset | Operator | Change | Impact |
|-------|----------|--------|--------|
| 13 | Softmax | axis semantics changed | Different outputs for same input |
| 13 | ReduceSum | axes: attribute → input | Graph structure change |
| 13 | Squeeze | axes: attribute → input | Graph structure change |
| 13 | Unsqueeze | axes: attribute → input | Graph structure change |
| 11 | Resize | coordinate_transform_mode | Interpolation behavior |
| 11 | Pad | pads: attribute → input | Graph structure change |

In [ ]:
# Demonstrate Softmax semantic difference between opset 12 and 13
# We build models at different opsets and compare outputs

input_shape = [1, 3, 4]
x_data = np.random.randn(*input_shape).astype(np.float32)

x_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, input_shape)
y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

results = {}
for opset in [11, 13, 17]:
    for axis in [1, -1]:
        try:
            node = helper.make_node("Softmax", ["X"], ["Y"], axis=axis)
            graph = helper.make_graph([node], f"sm_op{opset}_ax{axis}", [x_info], [y_info])
            model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])
            checker.check_model(model)

            from onnx.reference import ReferenceEvaluator
            ev = ReferenceEvaluator(model)
            y = ev.run(None, {"X": x_data})[0]
            results[(opset, axis)] = y
            print(f"Opset {opset:>2}, axis={axis:>2}: output shape={y.shape}, "
                  f"sum(axis=-1)[0]={y[0].sum(axis=-1).round(4)}")
        except Exception as e:
            print(f"Opset {opset:>2}, axis={axis:>2}: {str(e)[:60]}")

# Compare opset 11 vs 13 with axis=1
if (11, 1) in results and (13, 1) in results:
    r11 = results[(11, 1)]
    r13 = results[(13, 1)]
    same = np.allclose(r11, r13, atol=1e-5)
    print(f"\nOpset 11 vs 13 (axis=1) produce same output: {same}")
    if not same:
        print(f"  Max difference: {np.max(np.abs(r11 - r13)):.6f}")
        print(f"  This confirms the semantic change at opset 13!")

In [ ]:
# Demonstrate ReduceSum attribute-to-input change at opset 13

x_data = np.arange(24, dtype=np.float32).reshape(2, 3, 4)
print(f"Input shape: {x_data.shape}")
print(f"Expected ReduceSum(axis=1): {np.sum(x_data, axis=1).shape}")

# Opset 11: axes is an ATTRIBUTE
x_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 3, 4])
y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

try:
    node_11 = helper.make_node("ReduceSum", ["X"], ["Y"], axes=[1], keepdims=0)
    graph_11 = helper.make_graph([node_11], "rs_11", [x_info], [y_info])
    model_11 = helper.make_model(graph_11, opset_imports=[helper.make_opsetid("", 11)])
    checker.check_model(model_11)
    print(f"\nOpset 11: ReduceSum with axes as ATTRIBUTE — valid")
except Exception as e:
    print(f"\nOpset 11: {e}")

# Opset 13+: axes is an INPUT
try:
    axes_init = numpy_helper.from_array(np.array([1], dtype=np.int64), "axes")
    node_13 = helper.make_node("ReduceSum", ["X", "axes"], ["Y"], keepdims=0)
    graph_13 = helper.make_graph([node_13], "rs_13", [x_info], [y_info],
                                 initializer=[axes_init])
    model_13 = helper.make_model(graph_13, opset_imports=[helper.make_opsetid("", 13)])
    checker.check_model(model_13)
    print(f"Opset 13: ReduceSum with axes as INPUT — valid")

    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(model_13)
    y = ev.run(None, {"X": x_data})[0]
    expected = np.sum(x_data, axis=1)
    print(f"Result shape: {y.shape}, matches numpy: {np.allclose(y, expected)}")
except Exception as e:
    print(f"Opset 13: {e}")

<a id='6-version-converter'></a>
## 6. Version Converter

ONNX provides `onnx.version_converter.convert_version()` to **upgrade** or **downgrade**
models between opset versions.

### How It Works

The converter applies a chain of **adapters** — one per version step — that transform
nodes to match the target opset's schema:

```
  Model at opset 11 → convert_version(model, 17)

  Adapter 11→12:  update Clip inputs
  Adapter 12→13:  Softmax axis semantics, ReduceSum axes→input
  Adapter 13→14:  (no-op for most models)
  Adapter 14→15:  shape inference updates
  Adapter 15→16:  (no-op for most models)
  Adapter 16→17:  LayerNormalization promoted

  Result: Model at opset 17
```

### Limitations

- Not all conversions are possible (e.g., if an op was removed)
- **Downgrading** may fail if the model uses features not available in older opsets
- Custom domain ops are passed through unchanged
- The converter does NOT re-export from the framework — it transforms the ONNX graph

In [ ]:
# Demonstrate version conversion: upgrade and downgrade
np.random.seed(42)

# Build a model at opset 13
W = numpy_helper.from_array(np.random.randn(10, 5).astype(np.float32), name="W")
b = numpy_helper.from_array(np.random.randn(5).astype(np.float32), name="b")

nodes = [
    helper.make_node("MatMul", ["X", "W"], ["mm"]),
    helper.make_node("Add", ["mm", "b"], ["logits"]),
    helper.make_node("Softmax", ["logits"], ["Y"], axis=-1),
]
x_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 10])
y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 5])
graph = helper.make_graph(nodes, "convert_demo", [x_info], [y_info], initializer=[W, b])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
checker.check_model(model)

print(f"Original model: opset {model.opset_import[0].version}, ir={model.ir_version}")

# Upgrade to opset 17
try:
    upgraded = version_converter.convert_version(model, 17)
    checker.check_model(upgraded)
    print(f"Upgraded:       opset {upgraded.opset_import[0].version}, ir={upgraded.ir_version}")

    from onnx.reference import ReferenceEvaluator
    ev_orig = ReferenceEvaluator(model)
    ev_upgraded = ReferenceEvaluator(upgraded)
    x = np.random.randn(4, 10).astype(np.float32)
    y_orig = ev_orig.run(None, {"X": x})[0]
    y_upgraded = ev_upgraded.run(None, {"X": x})[0]
    print(f"Outputs match:  {np.allclose(y_orig, y_upgraded, atol=1e-6)}")
except Exception as e:
    print(f"Upgrade failed: {e}")

# Downgrade to opset 11
try:
    downgraded = version_converter.convert_version(model, 11)
    checker.check_model(downgraded)
    print(f"Downgraded:     opset {downgraded.opset_import[0].version}, ir={downgraded.ir_version}")

    ev_down = ReferenceEvaluator(downgraded)
    y_down = ev_down.run(None, {"X": x})[0]
    print(f"Outputs match:  {np.allclose(y_orig, y_down, atol=1e-5)}")
except Exception as e:
    print(f"Downgrade failed: {e}")

<a id='7-compatibility-triangle'></a>
## 7. Compatibility Triangle

Deployment requires matching three components: **model**, **runtime**, and **hardware**.

```
                      ┌───────────────────┐
                      │     ONNX Model     │
                      │   requires opset   │
                      │    V_model = 17    │
                      └─────────┬─────────┘
                                │
                    ╔═══════════╧═══════════╗
                    ║   COMPATIBILITY CHECK  ║
                    ║  V_model ≤ V_runtime?  ║
                    ╚═══════════╤═══════════╝
                                │
               ┌────────────────┴────────────────┐
               │                                  │
    ┌──────────▼──────────┐          ┌───────────▼──────────┐
    │  V_runtime ≥ 17     │          │   V_runtime < 17     │
    │  ✓ Compatible       │          │   ✗ Incompatible     │
    │  Load & run         │          │   Options:            │
    └─────────────────────┘          │   1. Upgrade runtime  │
                                     │   2. Downgrade model  │
                                     │   3. Re-export model  │
                                     └──────────────────────┘
```

### Compatibility Rules

| Condition | Result | Action |
|-----------|--------|--------|
| $V_{\text{model}} \leq V_{\text{runtime}}$ | Compatible | Load and run |
| $V_{\text{model}} > V_{\text{runtime}}$ | Incompatible | Downgrade model or upgrade runtime |
| Model uses custom domain | Depends | Runtime must have registered kernel |

### Forward vs Backward Compatibility

- **Backward compatible**: newer runtimes can run older models (guaranteed by ONNX spec)
- **Forward compatible**: older runtimes running newer models (NOT guaranteed — may fail)

In [ ]:
# Visualize the compatibility matrix
model_opsets = np.arange(7, 22)
runtime_opsets = np.arange(7, 22)

# Build compatibility matrix: 1 = compatible, 0 = incompatible
M, R = np.meshgrid(model_opsets, runtime_opsets)
compat = (M <= R).astype(float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = ax1.imshow(compat, cmap='RdYlGn', aspect='auto', origin='lower',
                extent=[model_opsets[0]-0.5, model_opsets[-1]+0.5,
                        runtime_opsets[0]-0.5, runtime_opsets[-1]+0.5])
ax1.set_xlabel('Model OpSet Version')
ax1.set_ylabel('Runtime Supported OpSet')
ax1.set_title('Compatibility Matrix\n(Green=Compatible, Red=Incompatible)', fontweight='bold')
ax1.plot([7, 21], [7, 21], 'k--', linewidth=1.5, alpha=0.5, label='V_model = V_runtime')
ax1.legend(loc='upper left')
plt.colorbar(im, ax=ax1, label='Compatible?')

# Version coverage by common runtimes
runtimes = {
    'ORT 1.12': 15,
    'ORT 1.14': 17,
    'ORT 1.16': 19,
    'ORT 1.18': 20,
    'TensorRT 8.5': 17,
    'OpenVINO 2023': 17,
}

y_pos = range(len(runtimes))
rt_names = list(runtimes.keys())
rt_versions = list(runtimes.values())

bars = ax2.barh(y_pos, rt_versions, color='steelblue', edgecolor='navy', alpha=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(rt_names)
ax2.set_xlabel('Max Supported OpSet Version')
ax2.set_title('Runtime OpSet Support', fontweight='bold')
ax2.set_xlim(0, 22)
for bar, ver in zip(bars, rt_versions):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2.,
             str(ver), va='center', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

<a id='8-multi-domain-models'></a>
## 8. Multi-Domain Models

A single ONNX model can import **multiple domains** simultaneously, each at its own
opset version. This is common when combining neural network operators (default domain)
with classical ML operators (`ai.onnx.ml`) or vendor extensions.

### Multi-Domain Architecture

```
  ┌─────────────────────────────────────────────────┐
  │  ModelProto                                      │
  │                                                  │
  │  opset_import: [{"":17}, {"ai.onnx.ml":3}]      │
  │                                                  │
  │  ┌──────────────────────────────────────────┐   │
  │  │ GraphProto                                │   │
  │  │                                           │   │
  │  │  MatMul(domain="")    ──▶ opset 17        │   │
  │  │        │                                  │   │
  │  │        ▼                                  │   │
  │  │  Relu(domain="")      ──▶ opset 17        │   │
  │  │        │                                  │   │
  │  │        ▼                                  │   │
  │  │  TreeEnsembleClassifier                   │   │
  │  │  (domain="ai.onnx.ml") ──▶ ML opset 3    │   │
  │  │                                           │   │
  │  └──────────────────────────────────────────┘   │
  └─────────────────────────────────────────────────┘
```

In [ ]:
# Build a multi-domain model (default + custom domain)
custom_domain = "com.tutorial.ops"

# Neural network part (default domain)
relu_node = helper.make_node("Relu", ["X"], ["relu_out"])
mm_node = helper.make_node("MatMul", ["relu_out", "W"], ["mm_out"])

# Custom domain operator
custom_node = helper.make_node("MyPostProcess", ["mm_out"], ["Y"],
                                domain=custom_domain)

# Model inputs/outputs
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 32])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 10])
W = numpy_helper.from_array(
    np.random.randn(32, 10).astype(np.float32), "W")

graph = helper.make_graph(
    [relu_node, mm_node, custom_node],
    "multi_domain", [X], [Y], initializer=[W])

# Multiple opset imports
model = helper.make_model(
    graph,
    opset_imports=[
        helper.make_opsetid("", 17),           # default domain
        helper.make_opsetid(custom_domain, 1),  # custom domain
    ])

print("Multi-domain model:")
print(f"  OpSet imports:")
for oi in model.opset_import:
    d = oi.domain if oi.domain else '"" (default)'
    print(f"    domain={d}, version={oi.version}")

print(f"\n  Nodes:")
for node in model.graph.node:
    d = node.domain if node.domain else '"" (default)'
    print(f"    {node.op_type:<20} domain={d}")

print(f"\n  Total size: {len(model.SerializeToString()):,} bytes")

<a id='9-ir-version-vs-opset-version'></a>
## 9. IR Version vs OpSet Version

ONNX has **two independent version axes** that are frequently confused:

| Property | IR Version | OpSet Version |
|----------|-----------|---------------|
| **What it versions** | Model structure (protobuf schema) | Operator semantics |
| **Scope** | Entire model format | Per-domain |
| **Controls** | Which fields exist in ModelProto, GraphProto, etc. | How operators behave |
| **Stored in** | `ModelProto.ir_version` | `ModelProto.opset_import` |
| **Updated when** | New proto fields added | New/changed operator schemas |
| **Current** | Changes less frequently | Changes with each ONNX release |

### Relationship Diagram

```
  ONNX Release 1.14
  ├── IR Version: 9
  │   └── Defines: ModelProto, GraphProto, NodeProto structure
  │
  ├── Default OpSet: 19
  │   └── Defines: Conv(v11), Softmax(v13), Gelu(v20), ...
  │
  └── ML OpSet: 3
      └── Defines: TreeEnsembleClassifier(v3), ...

  KEY: You can have IR version 8 with opset 17,
       or IR version 9 with opset 13.
       They are INDEPENDENT.
```

In [ ]:
# Show the mapping between ONNX versions, IR versions, and default opsets
print(f"Current installation:")
print(f"  ONNX library version: {onnx.__version__}")
print(f"  IR version:           {onnx.IR_VERSION}")
print(f"  Default opset:        {defs.onnx_opset_version()}")

# Build models with same opset but observe auto-assigned IR version
x = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 10])
y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 10])
node = helper.make_node("Relu", ["X"], ["Y"])
graph = helper.make_graph([node], "ir_demo", [x], [y])

print(f"\n{'OpSet':>6} │ {'Auto IR Version':>15} │ {'Model Size':>11}")
print("─" * 40)
for opset in [7, 9, 11, 13, 15, 17, 19, 20, 21]:
    try:
        m = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])
        size = len(m.SerializeToString())
        print(f"{opset:>6} │ {m.ir_version:>15} │ {size:>11}")
    except Exception as e:
        print(f"{opset:>6} │ {'error':>15} │ {str(e)[:20]}")

<a id='10-practical-checklist'></a>
## 10. Practical Checklist for OpSet Selection

When deciding which opset version to target, use this decision framework:

### Decision Flowchart

```
  START: Which opset should I use?
      │
      ▼
  ┌────────────────────────────────────┐
  │ Do you need specific operators     │
  │ (e.g., LayerNorm, GroupNorm)?      │
  └──────────┬───────────┬─────────────┘
             YES          NO
              │            │
              ▼            ▼
  ┌──────────────────┐  ┌──────────────────────┐
  │ Use minimum opset│  │ What is your target   │
  │ that has the op  │  │ runtime's max opset?  │
  │ (check since_ver)│  └──────────┬────────────┘
  └──────────────────┘             │
                                   ▼
                       ┌──────────────────────┐
                       │ Use that version or   │
                       │ the latest stable one  │
                       │ (typically opset 17-19)│
                       └──────────────────────┘
```

### Checklist

1. **Identify your minimum runtime** — What is the oldest ONNX Runtime (or other runtime) version in your fleet?
2. **Check runtime's max opset** — Map runtime version to maximum supported opset
3. **Check operator requirements** — Do you need ops introduced in newer opsets?
4. **Test with `checker.check_model()`** — Validate the model is well-formed
5. **Run `shape_inference.infer_shapes()`** — Verify shape propagation
6. **Test with ReferenceEvaluator** — Verify numerical correctness
7. **Test on target runtime** — Verify actual deployment compatibility
8. **Consider version converter** — If you need to support multiple opsets

### Common OpSet Recommendations

| Use Case | Recommended OpSet | Why |
|----------|------------------|-----|
| Maximum compatibility | 11–13 | Supported by nearly all runtimes |
| Modern features | 17 | LayerNorm, good type support |
| Cutting edge | 19–21 | Latest ops (GroupNorm, etc.) |
| Production (safe default) | 17 | Broad support, modern semantics |

In [ ]:
# Practical: check if a model is compatible with a target opset
def check_model_compatibility(model, target_opset):
    """Analyze if a model can run on a runtime supporting target_opset."""
    issues = []
    model_opset = model.opset_import[0].version

    if model_opset > target_opset:
        issues.append(f"Model opset ({model_opset}) > target ({target_opset})")

    for node in model.graph.node:
        domain = node.domain if node.domain else ''
        if domain == '' or domain == 'ai.onnx':
            try:
                schema = defs.get_schema(node.op_type, target_opset, '')
            except Exception:
                issues.append(f"{node.op_type} not available at opset {target_opset}")

    return issues

# Build a test model
x = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 3, 32, 32])
y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
w = numpy_helper.from_array(np.random.randn(16, 3, 3, 3).astype(np.float32)*0.1, "W")

nodes = [
    helper.make_node("Conv", ["X", "W"], ["conv_out"], kernel_shape=[3,3], pads=[1,1,1,1]),
    helper.make_node("Relu", ["conv_out"], ["Y"]),
]
graph = helper.make_graph(nodes, "compat_test", [x], [y], initializer=[w])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
checker.check_model(model)

# Check compatibility with different target opsets
print(f"Model built at opset 17")
print(f"Operators: {[n.op_type for n in model.graph.node]}")
print()

for target in [7, 9, 11, 13, 15, 17, 19]:
    issues = check_model_compatibility(model, target)
    status = '✓ Compatible' if not issues else '✗ Issues: ' + '; '.join(issues)
    print(f"  Target opset {target:>2}: {status}")

In [ ]:
# Visualize opset version timeline
all_schemas = defs.get_all_schemas_with_history()

version_data = {}
for s in all_schemas:
    if s.domain == '' or s.domain == 'ai.onnx':
        v = s.since_version
        if v not in version_data:
            version_data[v] = {'new': set(), 'updated': set()}
        version_data[v]['new'].add(s.name)

versions = sorted(version_data.keys())
new_counts = [len(version_data[v]['new']) for v in versions]
cumulative = np.cumsum(new_counts)

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(versions, new_counts, color='steelblue', edgecolor='navy', alpha=0.8,
       label='New/Updated schemas')
ax2 = ax.twinx()
ax2.plot(versions, cumulative, 'r-o', markersize=5, linewidth=2,
         label='Cumulative total')

ax.set_xlabel('OpSet Version', fontsize=12)
ax.set_ylabel('Schemas in this version', fontsize=12, color='steelblue')
ax2.set_ylabel('Cumulative schemas', fontsize=12, color='red')
ax.set_title('ONNX Default Domain: Schema Evolution Over OpSet Versions',
             fontsize=13, fontweight='bold')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

<a id='11-key-takeaways'></a>
## 11. Key Takeaways

1. An **OpSet** is a versioned, domain-scoped collection of operator schemas:
   $(\texttt{domain}, \texttt{version}) \mapsto \{\text{schemas}\}$.

2. Models declare opset imports via `ModelProto.opset_import`, and each node's behavior
   is resolved from the imported version for its domain.

3. **Version resolution** picks the highest schema version $\leq$ the imported opset:
   $$v_{\text{eff}}(t, d) = \max\{v_s \mid v_s \leq v_d \wedge \text{schema\_exists}(t, v_s, d)\}$$

4. **Semantic changes** can occur between opset versions. The most significant was
   opset 13, which changed Softmax axis behavior and moved ReduceSum/Squeeze axes
   from attributes to inputs.

5. **IR version** and **opset version** are independent axes. IR version controls the
   model structure format; opset version controls operator semantics.

6. The **version converter** can upgrade or downgrade models between opsets, but not
   all conversions are possible.

7. **Backward compatibility** is guaranteed (newer runtimes run older models);
   **forward compatibility** is not (older runtimes may fail on newer models).

8. For production, **opset 17** is a safe default with broad runtime support and
   modern operator semantics.